### **Spatial Join Notebook:**
This notebook prepares the spatial connection between the cleaned ZüriWieNeu report data and the neighborhood boundaries of the city of Zurich. The main objective is to assign each reported urban issue to a specific neighbourhood in Zurich. This is an important step because it allows the reports to be analysed not only as individual point locations, but also at neighbourhood level.

First, the cleaned report dataset is loaded and converted into a GeoDataFrame. The coordinate columns are used to create point geometries, and the coordinate reference system is set to EPSG:2056, which is suitable for spatial data in Switzerland.

Second, the Zurich neighbourhood data are loaded from the GeoPackage. The layer used in this notebook is stzh.adm_statistische_quartiere_v, because it represents the neighbourhoods as polygon geometries. The neighbourhood layer is inspected and cleaned by keeping only the relevant columns and renaming them so that they are easier to interpret.

Finally, a spatial join is performed between the report points and the neighbourhood polygons. The result is a joined GeoDataFrame in which each report is assigned to a Zurich neighbourhood. By the end of this notebook, the joined dataset is ready to be saved and used for further spatial and temporal analysis.

Expected output:
- a GeoDataFrame containing all cleaned reports
- point geometries for each report
- neighbourhood information added to each report
- a processed GeoPackage that can be used in the analysis notebook

---
Firstly the necessary libary for this notebook were imported. Pandas is needed to import the cleaned CSV Dataset of the Reports and geopandas is needed to load a GeoDataFrame.

In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
zh_reports_clean = pd.read_csv("../data/processeddata/zueriwieneu_cleaned.csv", index_col="service_request_id")

After cleaning the data and adding new useful columns, the CSV file is converted into a GeoDataFrame so that it can be used together with the Zurich neighbourhood data. The neighbourhood GeoPackage uses the coordinate reference system EPSG:2056 (information from while downloading the dataset, but can be easily checked in python aswell), so the reports GeoDataFrame is set to the same coordinate system. The coordinate columns e and n are given in meters, which also fits this coordinate system. This information was taken from the metadata, but it will be checked again in the following steps.

In [3]:
zh_reports_gdf = gpd.GeoDataFrame(
    zh_reports_clean,
    geometry=gpd.points_from_xy(zh_reports_clean["e"], zh_reports_clean["n"]),
    crs="EPSG:2056"
)

To spatially analyse the data, a spatial join is needed because the reports GeoDataFrame does not contain a neighbourhood column that could be used to merge the datasets directly. First, it is checked whether the reports data was successfully converted into a GeoDataFrame and whether the report locations are represented as geometries.

In [4]:
zh_reports_gdf[["e", "n", "geometry"]].head(2)

,e,n,geometry
service_request_id,,,
1,2678968,1247548,POINT (2678968 1247548)
2,2680746,1249916,POINT (2680746 1249916)


With only looking at the fisrt 2 rows, it is confrimed that the Reports Data set was corrcetly loaded in a GeoDataFrame and the coordinates are representing a geogemtry.

The neighbourhood dataset is loaded and inspected first. It is important to load the specific layer stzh.adm_statistische_quartiere_v, because this layer represents the neighbourhoods as polygons rather than points. Polygon geometries are needed for the spatial join, since the report points can then be assigned to the neighbourhood area in which they are located.

In [5]:
zh_quartiere = gpd.read_file("../data/rawdata/data/quartier_data.gpkg", layer="stzh.adm_statistische_quartiere_v")

To check whether the correct data layer was loaded, the geometry type and the number of features are inspected. The layer should contain polygon geometries, because these are needed to assign the report points to neighbourhood areas.

In [6]:
zh_quartiere.geom_type.value_counts()

Polygon    34
Name: count, dtype: int64

In [7]:
#getting a first look at the layer and making sure they are sharing the same crs
zh_quartiere.info()
zh_quartiere.crs

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   objid     34 non-null     str     
 1   objectid  34 non-null     int32   
 2   qname     34 non-null     str     
 3   qnr       34 non-null     int32   
 4   kname     34 non-null     str     
 5   knr       34 non-null     int32   
 6   geometry  34 non-null     geometry
dtypes: geometry(1), int32(3), str(3)
memory usage: 1.6 KB


<Projected CRS: EPSG:2056>
Name: CH1903+ / LV95
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Liechtenstein; Switzerland.
- bounds: (5.95, 45.81, 10.5, 47.81)
Coordinate Operation:
- name: Swiss Oblique Mercator 1995
- method: Hotine Oblique Mercator (variant B)
Datum: CH1903+
- Ellipsoid: Bessel 1841
- Prime Meridian: Greenwich

After reviewing the variables in the dataset, it becomes clear that not all variables are needed for the analysis. Additionally, the variable names are should get cleaned to make it easier to understand what each variable represents, especially after the spatial join. First, a copy of the dataset is created to avoid changing the original input data during the cleaning process. Afterwards, the variables are renamed using a dictionary that maps the original names to clearer names. Finally, a list of relevant columns is defined and kept in the dataset. The result is then checked with the dataset information to confirm that the cleaning process worked correctly.

In [8]:
#cleaining the column names in the quartiere dataset after making a copy, to not change the original dataset
zh_quartiere_clean = zh_quartiere.copy()

zh_quartiere_clean = zh_quartiere_clean.rename(
    columns={
        "qname": "quartier_name",
        "qnr": "quartier_number",
        "kname": "kreis_name",
        "knr": "kreis_number"
    }
)

zh_quartiere_clean = zh_quartiere_clean[
    [
        "quartier_name",
        "quartier_number",
        "kreis_name",
        "kreis_number",
        "geometry"
    ]
]
zh_quartiere_clean.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   quartier_name    34 non-null     str     
 1   quartier_number  34 non-null     int32   
 2   kreis_name       34 non-null     str     
 3   kreis_number     34 non-null     int32   
 4   geometry         34 non-null     geometry
dtypes: geometry(1), int32(2), str(2)
memory usage: 1.2 KB


The info output shows, that the names were renamed correctly and there are only the useful varibales left.



Now that the dataset has been cleaned, it is saved in the processed data folder. This makes it possible to reload the cleaned Quartiere dataset later, for example for visualizations or further analysis, without repeating the cleaning steps.

In [9]:
zh_quartiere_clean.to_file(
    "../data/processeddata/zh_quartiere_clean.gpkg",
    layer="zh_quartiere_clean",
    driver="GPKG"
)

Next, the spatial join between the cleaned reports geodata frame and the cleaned Quartiere data set, so every Report Data point gets assigned a Quartier in Zurich. This next step is the crucial preparation for the analysis later on.

In [10]:
#The follwing code takes the reports geo dataframe and matches it to the quartiere, the point coordinate lies within. 
#With within, each point only gets connecetd to one Quartiere Polygon, and not to multiple, if it interects with two polygons.
reports_with_quartiere = gpd.sjoin(
    zh_reports_gdf,
    zh_quartiere_clean,
    how="left",
    predicate="within"
)

To check whether the spatial join was successful, the first two rows an the general information of the joined dataset are inspected.

In [11]:
reports_with_quartiere.head(2)

,requested_datetime,updated_datetime,e,n,service_code,status,detail,year_requested,month_requested,processing_time_days,geometry,index_right,quartier_name,quartier_number,kreis_name,kreis_number
service_request_id,,,,,,,,,,,,,,,,
1,2013-03-14 15:16:15,2013-04-12 07:59:30,2678968,1247548,Strasse/Trottoir/Platz,fixed - council,Auf dem Asphalt des Bürgersteigs hat es eine E...,2013,3,28.696701,POINT (2678968 1247548),16,Albisrieden,91,Kreis 9,9
2,2013-03-14 15:17:57,2013-04-12 08:00:22,2680746,1249916,Strasse/Trottoir/Platz,fixed - council,Vermessungspunkt ist nicht mehr bündig mit dem...,2013,3,28.696123,POINT (2680746 1249916),20,Höngg,101,Kreis 10,10


In [12]:
reports_with_quartiere.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 72606 entries, 1 to 81194
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   requested_datetime    72606 non-null  str     
 1   updated_datetime      72606 non-null  str     
 2   e                     72606 non-null  int64   
 3   n                     72606 non-null  int64   
 4   service_code          72606 non-null  str     
 5   status                72606 non-null  str     
 6   detail                72604 non-null  str     
 7   year_requested        72606 non-null  int64   
 8   month_requested       72606 non-null  int64   
 9   processing_time_days  72606 non-null  float64 
 10  geometry              72606 non-null  geometry
 11  index_right           72606 non-null  int64   
 12  quartier_name         72606 non-null  str     
 13  quartier_number       72606 non-null  int32   
 14  kreis_name            72606 non-null  str     
 15 

The output shows that the spatial join worked successfully. The joined dataset contains 81194 reports and 20 columns. One thing to note is that the date columns are stored as string values again. This likely happened because the data was saved and loaded again. However, the most important time-related columns, such as year, month and processing time in days, have already been calculated. Therefore, it is not necessary to convert the date columns back to datetime at this stage. If further time-based analysis is needed later, for example changes over  time or filtering by date, the date columns can be converted again in the analysis notebook.

An important quality check is whether every report point in the joined dataset was assigned to a neighbourhood.

In [13]:
#checking how many reports did not recieve a quartiere and show the sum of not assinged reports
reports_with_quartiere["quartier_name"].isna().sum()

np.int64(0)

This output shows, that there are zero reports not matched to a Quartier. So every report is spatially assinged to a Quartier.

**saving the GeoDataFrame:** After the spatial Join and checking if it worked, the Dataset is saved in the procceses data folder as a Geopackage, so the geometry still is availabe.

In [14]:
reports_with_quartiere.to_file(
    "../data/processeddata/reports_with_quartiere.gpkg",
    layer="reports_with_quartiere",
    driver="GPKG"
)